In [2]:
# LDA分类器
import pandas as pd
from nlpia.data.loaders import get_data
pd.options.display.width = 120
sms = get_data('sms-spam')
index = ['sms{}{}'.format(i, '!'*j) for (i, j) in zip(range(len(sms)), sms.spam)]
sms = pd.DataFrame(sms.values, columns=sms.columns, index=index)
len(sms) #4837
sms.spam.sum() #638
sms.head(6)
#       spam                                               text
# sms0     0  Go until jurong point, crazy.. Available only ...
# sms1     0                      Ok lar... Joking wif u oni...
# sms2!    1  Free entry in 2 a wkly comp to win FA Cup fina...
# sms3     0  U dun say so early hor... U c already then say...
# sms4     0  Nah I don't think he goes to usf, he lives aro...
# sms5!    1  FreeMsg Hey there darling it's been 3 week's n...

,spam,text
sms0,0,"Go until jurong point, crazy.. Available only ..."
sms1,0,Ok lar... Joking wif u oni...
sms2!,1,Free entry in 2 a wkly comp to win FA Cup fina...
sms3,0,U dun say so early hor... U c already then say...
sms4,0,"Nah I don't think he goes to usf, he lives aro..."
sms5!,1,FreeMsg Hey there darling it's been 3 week's n...


In [13]:
# 分词
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize.casual import casual_tokenize
tfidf_model = TfidfVectorizer(tokenizer=casual_tokenize)
tfidf_docs = tfidf_model.fit_transform(raw_documents=sms.text).toarray()
tfidf_docs.shape # (4837, 9232)

/Users/maverick/miniforge3/envs/studynote/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


(4837, 9232)

In [17]:
from sklearn.decomposition import PCA
pca = PCA(n_components=16)
pca = pca.fit(tfidf_docs)
pca_topic_vectors = pca.transform(tfidf_docs)
pca_topic_vectors.shape # (4837, 16)
columns = ['topic{}'.format(i) for i in range(pca_topic_vectors.shape[1])]
pca_topic_vectors = pd.DataFrame(pca_topic_vectors, columns=columns, index=index)
pca_topic_vectors.head(6)

,topic0,topic1,topic2,topic3,topic4,topic5,topic6,topic7,topic8,topic9,topic10,topic11,topic12,topic13,topic14,topic15
sms0,0.201175,-0.002826,-0.037225,0.010920,-0.019218,-0.053041,-0.039140,-0.066047,0.014456,-0.084061,0.006405,-0.020197,-0.002772,-0.042812,-0.008709,-0.006305
sms1,0.404379,0.093891,0.077509,0.050902,0.100101,0.047115,-0.022942,0.064874,0.022627,-0.023948,-0.003332,0.033771,0.047372,-0.017660,0.048757,0.046553
sms2!,-0.030453,0.048093,-0.090178,-0.067125,0.090743,-0.043311,0.000054,-0.000673,-0.054448,0.049790,0.129023,0.002353,0.016270,-0.014046,-0.030377,-0.013276
sms3,0.329047,0.032802,0.034551,-0.015779,0.052191,0.055646,0.165514,-0.075107,0.061360,-0.107866,0.025485,0.013809,0.072965,-0.038781,0.016384,0.045491
sms4,0.002160,-0.030879,-0.038315,0.033847,-0.074678,-0.092500,0.043763,0.062245,-0.044832,0.028294,0.025248,-0.009480,0.027703,0.038454,-0.083707,0.015235
sms5!,-0.015976,-0.058601,-0.014002,-0.006115,0.122354,-0.040070,-0.005009,0.167888,-0.021446,0.061787,0.037297,0.051612,-0.030525,0.078905,0.008401,-0.034606


In [20]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=16, n_iter=7)
tfidf_docs = pd.DataFrame(tfidf_docs)
tfidf_docs = tfidf_docs - tfidf_docs.mean()
svd_topic_vectors = svd.fit_transform(tfidf_docs.values)
svd_topic_vectors = pd.DataFrame(svd_topic_vectors, columns=columns, index=index)
svd_topic_vectors.round(3).head(6)

,topic0,topic1,topic2,topic3,topic4,topic5,topic6,topic7,topic8,topic9,topic10,topic11,topic12,topic13,topic14,topic15
sms0,0.201,-0.003,-0.037,0.011,-0.019,-0.053,-0.039,-0.065,0.012,-0.084,0.007,-0.005,0.004,-0.036,0.004,0.034
sms1,0.404,0.094,0.078,0.051,0.100,0.047,-0.023,0.063,0.023,-0.023,-0.004,0.034,0.046,-0.020,0.042,-0.041
sms2!,-0.030,0.048,-0.090,-0.067,0.091,-0.043,0.000,0.001,-0.058,0.050,0.124,0.022,0.024,-0.023,-0.037,0.052
sms3,0.329,0.033,0.035,-0.016,0.052,0.056,0.165,-0.077,0.064,-0.106,0.024,0.021,0.073,-0.048,0.017,-0.056
sms4,0.002,-0.031,-0.038,0.034,-0.075,-0.093,0.044,0.060,-0.045,0.028,0.029,-0.006,0.033,0.032,-0.082,-0.018
sms5!,-0.016,-0.059,-0.014,-0.006,0.122,-0.040,-0.005,0.167,-0.025,0.061,0.041,0.059,-0.032,0.077,0.009,0.024


In [21]:
import numpy as np
svd_topic_vectors = (svd_topic_vectors.T / np.linalg.norm(svd_topic_vectors, axis=1)).T
svd_topic_vectors.iloc[:10].dot(svd_topic_vectors.iloc[:10].T).round(1)

,sms0,sms1,sms2!,sms3,sms4,sms5!,sms6,sms7,sms8!,sms9!
sms0,1.0,0.7,-0.1,0.7,-0.0,-0.3,-0.3,-0.1,-0.3,-0.3
sms1,0.7,1.0,-0.1,0.8,-0.2,0.0,-0.2,-0.2,-0.1,-0.1
sms2!,-0.1,-0.1,1.0,-0.1,0.1,0.4,0.1,0.3,0.5,0.4
sms3,0.7,0.8,-0.1,1.0,-0.2,-0.3,-0.1,-0.3,-0.2,-0.1
sms4,-0.0,-0.2,0.1,-0.2,1.0,0.2,0.0,0.1,-0.4,-0.2
sms5!,-0.3,0.0,0.4,-0.3,0.2,1.0,-0.1,0.1,0.3,0.4
sms6,-0.3,-0.2,0.1,-0.1,0.0,-0.1,1.0,0.1,-0.2,-0.2
sms7,-0.1,-0.2,0.3,-0.3,0.1,0.1,0.1,1.0,0.1,0.4
sms8!,-0.3,-0.1,0.5,-0.2,-0.4,0.3,-0.2,0.1,1.0,0.3
sms9!,-0.3,-0.1,0.4,-0.1,-0.2,0.4,-0.2,0.4,0.3,1.0


In [32]:
# LDiA
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import casual_tokenize
np.random.seed(42)

counter = CountVectorizer(tokenizer=casual_tokenize)
bow_docs = pd.DataFrame(counter.fit_transform(raw_documents=sms.text).toarray(), index=index)
column_nums, terms = zip(*sorted(zip(counter.vocabulary_.values(), counter.vocabulary_.keys())))
bow_docs.columns = terms

sms.loc['sms0'].text
bow_docs.loc['sms0'][bow_docs.loc['sms0'] > 0].head()

,            1
..           1
...          2
amore        1
available    1
Name: sms0, dtype: int64

In [33]:
from sklearn.decomposition import LatentDirichletAllocation as LDiA
ldia = LDiA(n_components=16, learning_method='batch')
ldia = ldia.fit(bow_docs)
ldia.components_.shape # (16, 9232)

(16, 9232)

In [34]:
pd.set_option('display.width', 75)
components = pd.DataFrame(ldia.components_.T, index=terms, columns=columns)
components.round(2).head(3)

,topic0,topic1,topic2,topic3,topic4,topic5,topic6,topic7,topic8,topic9,topic10,topic11,topic12,topic13,topic14,topic15
!,184.03,15.00,72.22,394.95,45.48,36.14,9.55,44.81,0.43,90.23,37.42,44.18,64.40,297.29,41.16,11.70
"""",0.68,4.22,2.41,0.06,152.35,0.06,0.06,0.06,0.45,0.68,8.42,11.42,0.07,62.72,12.27,0.06
#,0.06,0.06,0.06,0.06,0.06,2.07,0.06,0.06,0.06,0.06,0.06,0.06,1.07,4.05,0.06,0.06


In [35]:
ldia16_topic_vectors = ldia.transform(bow_docs)
ldia16_topic_vectors = pd.DataFrame(ldia16_topic_vectors, columns=columns, index=index)
ldia16_topic_vectors.round(2).head()

,topic0,topic1,topic2,topic3,topic4,topic5,topic6,topic7,topic8,topic9,topic10,topic11,topic12,topic13,topic14,topic15
sms0,0.00,0.62,0.00,0.00,0.00,0.00,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.00,0.00,0.00
sms1,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.78,0.01,0.01,0.12,0.01,0.01,0.01,0.01
sms2!,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.98,0.00,0.00,0.00,0.00,0.00,0.00
sms3,0.00,0.00,0.00,0.00,0.09,0.00,0.00,0.00,0.85,0.00,0.00,0.00,0.00,0.00,0.00,0.00
sms4,0.39,0.00,0.33,0.00,0.00,0.00,0.14,0.00,0.00,0.00,0.00,0.00,0.09,0.00,0.00,0.00


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
y = sms.spam.astype(int)
X_train, X_test, y_train, y_test = train_test_split(ldia16_topic_vectors, y, test_size=0.5, random_state=271828)
lda = LDA(n_components=1)
lda = lda.fit(X_train, y_train)
sms['ldia16_spam'] = lda.predict(ldia16_topic_vectors)
round(float(lda.score(X_test, y_test)), 2) # 0.94

0.94